In [2]:
import pandas as pd
import os
from sas7bdat import SAS7BDAT
from deep_translator import GoogleTranslator
import numpy as np

In [3]:
# import argostranslate.package
# import argostranslate.translate

# argostranslate.package.update_package_index()
# packages = argostranslate.package.get_available_packages()

# # Install all → English packages
# for p in packages:
#     if p.to_code == "en":
#         argostranslate.package.install_from_path(p.download())

In [4]:
import pandas as pd
import numpy as np
import os
from tqdm import tqdm
import math
import re

import sys
sys.path.append('../../..')
# from mount_drive import mount_s_drive

In [5]:
# mount_s_drive(subfolder='LCICM/Databases/cardiac_arrest_nantes/Donnees a transferer/')

In [6]:
database_folder = '/projects/LCICM/cardiac_arrest_nantes/Donnees a transferer/'

### Read data from copy of hyperion dataset, and convert all files into a dataframe

In [7]:
read_path = database_folder

In [8]:
myDfs = {}
for file in os.listdir(read_path):
    if 'sas7bdat' in file:
        with SAS7BDAT(read_path + file) as reader:
            myDfs[file.split('.')[0]] = reader.to_data_frame()
        
    if 'xlsx' in file: 
        myDfs[file.split('.')[0]] = pd.read_excel(read_path + file)

In [9]:
myDfs.keys()

dict_keys(['Description_tables_variables', 'vj', 'ds', 'ecg', 'adl', 'barthel', 'sofa', 'cpc', 'ies', 'pi', 'j0', 'date', 'h0h64', 'ei', 'mms', 'pavm', 'sf36', 'v0', 'sh', 'bras_rando', 'm3', 'ttj0', 'ir', 'bio'])

In [10]:
for key, value in myDfs.items():
    print(f'the data: {key}')
    print(value.head().to_string())
    print()

the data: Description_tables_variables
  Tables    Variable                                                  Label  Type  Lenght  Number    Format
0    ADL      SUBJID                       Subject Identifier for the Study  char      10       1      $10.
1    ADL       ADL_0                       Le score ADL a t-il été complété   num       8       2   OUINON.
2    ADL  ADL_1_TEMP   Hygiène corporelle (champ intermediaire pour saisie)   num       8       3  T_ADL1_.
3    ADL  ADL_2_TEMP            Habillage (champ intermediaire pour saisie)   num       8       4  T_ADL2_.
4    ADL  ADL_3_TEMP  Aller aux toilettes (champ intermediaire pour saisie)   num       8       5  T_ADL3_.

the data: vj
  SUBJID VISIT VJ_VISITE       VJ_DT_DEBUT         VJ_DT_FIN  VJ_GLASGOW  VJ_OCULAIRE  VJ_VERBALE  VJ_MOTRICE  VJ_PUPILG VJ_PUPILG_REA  VJ_PUPILD VJ_PUPILD_REA  VJ_CILIAIRE  VJ_CORNEEN  VJ_REFLEXVEST  VJ_REFLEXCEPH  VJ_REFLEXCARD  VJ_VENTIL  VJ_TEMP_MIN  VJ_TEMP_MAX  VJ_HYPNO  VJ_HYPNO_TRT  VJ_MORP

### Convert all labels to english

In [14]:
import argostranslate.package
import argostranslate.translate

def install_local_model(argosmodel_path: str):
    # e.g., "/path/to/ar_en.argosmodel"
    argostranslate.package.install_from_path(argosmodel_path)

def tryConvert(x: str) -> str:
    try:
        # pick a fixed source if you know it; "auto" is not reliably supported offline
        return argostranslate.translate.translate(x, from_code="fr", to_code="en")
    except Exception:
        return x

myLambda = lambda x: tryConvert(x.Label)
myDfs['Description_tables_variables']['LabelTranslated'] = myDfs['Description_tables_variables'].apply(myLambda, axis=1)

In [12]:
print(myDfs['Description_tables_variables'][['Tables', 'Variable', 'LabelTranslated']].to_string())

         Tables                          Variable                                                                                       LabelTranslated
0           ADL                            SUBJID                                                                      Subject Identifier for the Study
1           ADL                             ADL_0                                                                      Le score ADL a t-il été complété
2           ADL                        ADL_1_TEMP                                                  Hygiène corporelle (champ intermediaire pour saisie)
3           ADL                        ADL_2_TEMP                                                           Habillage (champ intermediaire pour saisie)
4           ADL                        ADL_3_TEMP                                                 Aller aux toilettes (champ intermediaire pour saisie)
5           ADL                        ADL_4_TEMP                                       

### Convert all files to csvs

In [15]:
# argostranslate.translate.translate('Unité leucocytes', from_code="fr", to_code="en")

In [34]:
write_path = './formatteddata/'
for key, value in myDfs.items():
    value.to_csv(write_path + key + '.csv')
    

In [37]:
myDfs['Description_tables_variables']['LabelTranslated']

0                       Subject Identifier for the Study
1                       Le score ADL a t-il été complété
2      Hygiène corporelle (champ intermediaire pour s...
3            Habillage (champ intermediaire pour saisie)
4      Aller aux toilettes (champ intermediaire pour ...
                             ...                        
711                                        Antipyrétique
712                                          Paracétamol
713                                             Aspirine
714                                                  NaN
715                                                  NaN
Name: LabelTranslated, Length: 716, dtype: object